In [598]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.font_manager as fm
import matplotlib
import lightgbm as lgb
from xgboost import XGBClassifier
from sklearn.model_selection import cross_validate
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import PolynomialFeatures
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder

import re

font_path = "C:/Windows/Fonts/gulim.ttc"
font = fm.FontProperties(fname=font_path).get_name()
matplotlib.rc("font", family=font)

In [599]:
team_df = pd.read_csv("../../EDA/Merge/data/result data/Final DF.csv")
team_df

,Year,Nation,Eng_Nation,Wc_Rank,Wc_Point,Q_WR,Q_GR,F_Rank,F_Point,F_Rd,...,FS_7,FS_8,FS_9,FS_10,FS_11,FS_12,FS_13,FS_14,ATK_INDEX,DEF_INDEX
0,2002,브라질,Brazil,1,21,0.46,2.352113,1.33,818.33,0.00,...,75.27,77.05,72.73,82.09,31.95,68.95,59.61,62.32,0.141500,-0.070093
1,2002,독일,Germany,2,16,0.55,1.817073,8.67,718.33,0.00,...,66.38,67.86,72.57,80.57,37.60,62.88,63.29,63.48,0.031444,0.011481
2,2002,터키,Turkey,3,13,0.46,1.457627,33.00,593.33,0.00,...,66.45,69.80,67.05,74.05,30.85,65.18,56.78,57.60,-0.039941,0.015588
3,2002,대한민국,South Korea,4,11,0.48,2.216667,42.00,573.00,0.00,...,65.91,65.50,65.68,72.45,32.64,57.66,62.23,57.09,0.040250,0.133021
4,2002,스페인,Spain,5,11,0.65,3.227273,5.00,742.33,0.00,...,75.50,78.23,78.09,83.27,35.05,69.80,64.55,71.05,-0.105762,-0.099683
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
187,2022,덴마크,Denmark,28,1,0.61,2.446429,13.33,1611.83,-18.00,...,72.19,70.69,67.92,70.15,17.13,66.37,53.93,55.04,NaN,NaN
188,2022,세르비아,Serbia,29,1,0.48,1.536232,31.33,1492.66,-18.00,...,66.19,72.58,69.38,64.15,18.21,64.19,51.35,54.31,NaN,NaN
189,2022,웨일스,Wales,30,1,0.44,1.233333,21.00,1545.20,10.75,...,66.19,63.88,55.19,66.27,17.62,59.62,53.12,45.12,NaN,NaN
190,2022,캐나다,Canada,31,0,0.48,2.296875,66.33,1356.74,-34.00,...,66.48,69.76,60.36,65.60,16.50,56.82,43.71,47.96,NaN,NaN


In [600]:
len(team_df)

192

In [601]:
import random


def make_group(df, teams_per_group=4, seed=None):
    if seed is not None:
        random.seed(seed)

    n_teams = len(df)
    n_groups = n_teams // teams_per_group
    groups = {}
    indices = df.index.tolist()
    random.shuffle(indices)

    for i in range(n_groups):
        group_name = chr(ord("A") + i)  # 'A', 'B', 'C', ...
        groups[group_name] = indices[i * teams_per_group : (i + 1) * teams_per_group]

    return groups

In [602]:
groups = make_group(team_df[:32], seed=42)

In [603]:
def get_match_result(home, away):
    return np.random.randint(-1, 2, 1)


get_match_result("a", "b")

array([0], dtype=int32)

In [604]:
groups

{'A': [26, 5, 10, 15],
 'B': [25, 11, 22, 6],
 'C': [19, 12, 16, 9],
 'D': [28, 14, 24, 20],
 'E': [30, 1, 13, 18],
 'F': [2, 17, 21, 3],
 'G': [29, 4, 27, 31],
 'H': [8, 23, 0, 7]}

In [605]:
from itertools import combinations

list(combinations(groups["A"], 2))

[(26, 5), (26, 10), (26, 15), (5, 10), (5, 15), (10, 15)]

In [606]:
from itertools import combinations


def do_groupstage(team_df, groups):
    result = {}
    if len(team_df) == 32:
        n_result = 16
    elif len(team_df) == 48:
        n_result = 32
        thrid_lst = {}
    for g in groups.keys():
        matches = {}
        for idx in groups[g]:
            matches[idx] = 0
        for home, away in list(combinations(groups[g], 2)):
            match_result = get_match_result(team_df.iloc[home], team_df.iloc[away])
            if match_result == 1:
                matches[home] += 3
            elif match_result == -1:
                matches[away] += 3
            else:
                matches[home] += 1
                matches[away] += 1
        result[g] = sorted(matches, key=lambda x: matches[x])[:3]
        if n_result == 32:
            thrid_lst[result[g][2]] = [matches[result[g][2]], g]
        result[g] = result[g][:2]
    if n_result == 32:
        sorted_lst = sorted(thrid_lst, key=lambda x: thrid_lst[x][0], reverse=True)[:8]
        for i in sorted_lst:
            result[thrid_lst[i][1]].append(i)
            pass
    return result

In [607]:
X = team_df[:48]
groups = make_group(X)
result_group = do_groupstage(X, groups)
result_group

{'A': [0, 3, 10],
 'B': [30, 8, 14],
 'C': [34, 4, 39],
 'D': [42, 38, 35],
 'E': [31, 25],
 'F': [37, 11],
 'G': [19, 20, 47],
 'H': [33, 12],
 'I': [2, 16, 29],
 'J': [5, 40, 24],
 'K': [13, 6],
 'L': [27, 44, 17]}

In [608]:
def do_sort(team_df, groups):
    tornament = []
    for g in groups.keys():
        groups[g].sort(reverse=True)
    return groups


result_group = do_sort(X, result_group)
result_group

{'A': [10, 3, 0],
 'B': [30, 14, 8],
 'C': [39, 34, 4],
 'D': [42, 38, 35],
 'E': [31, 25],
 'F': [37, 11],
 'G': [47, 20, 19],
 'H': [33, 12],
 'I': [29, 16, 2],
 'J': [40, 24, 5],
 'K': [13, 6],
 'L': [44, 27, 17]}

In [609]:
'''
[조별리그 이후 토너먼트 배치 함수]
- 48개팀은 4개팀씩 12개조로 편성
- 팀당 3경기를 치러 각 조 1, 2위 24개팀은 조별리그를 통과
- 3위 중 가장 좋은 성적을 거둔 8개 팀도 토너먼트에 합류

- 토너먼트 32강에서 12개조 1위팀들 중 8팀은 다른 조 3위팀과 경기
- (조 3위팀의 32강전 상대는 조 1위 중 성적이 가장 좋은 8개 팀 중 하나)
- 나머지 1위 4팀은 2위 4팀과 경기
- 1위팀과 경기하지 않는 나머지 2위 8팀이 4팀씩 경기

# 32강 # 아래 순서대로 리스트에 # vs는 무시하고 다음 인덱스 # 예: [0]vs[1]
round_32 = [    
    E조 1위 vs A/B/C/D/F조 3위
    I조 1위 vs C/D/F/G/H조 3위

    A조 2위 vs B조 2위
    F조 1위 vs C조 2위

    C조 1위 vs F조 2위
    E조 2위 vs I조 2위

    A조 1위 vs C/E/F/H/I조 3위
    L조 1위 vs E/H/I/J/K조 3위

    K조 2위 vs L조 2위
    H조 1위 vs J조 2위

    D조 1위 vs B/E/F/I/J조 3위
    G조 1위 vs A/E/H/I/J조 3위

    J조 1위 vs H조 2위
    D조 2위 vs G조 2위

    B조 1위 vs E/F/G/I/J조 3위
    K조 1위 vs D/E/I/J/L조 3위
]
'''

'\n[조별리그 이후 토너먼트 배치 함수]\n- 48개팀은 4개팀씩 12개조로 편성\n- 팀당 3경기를 치러 각 조 1, 2위 24개팀은 조별리그를 통과\n- 3위 중 가장 좋은 성적을 거둔 8개 팀도 토너먼트에 합류\n\n- 토너먼트 32강에서 12개조 1위팀들 중 8팀은 다른 조 3위팀과 경기\n- (조 3위팀의 32강전 상대는 조 1위 중 성적이 가장 좋은 8개 팀 중 하나)\n- 나머지 1위 4팀은 2위 4팀과 경기\n- 1위팀과 경기하지 않는 나머지 2위 8팀이 4팀씩 경기\n\n# 32강 # 아래 순서대로 리스트에 # vs는 무시하고 다음 인덱스 # 예: [0]vs[1]\nround_32 = [    \n    E조 1위 vs A/B/C/D/F조 3위\n    I조 1위 vs C/D/F/G/H조 3위\n\n    A조 2위 vs B조 2위\n    F조 1위 vs C조 2위\n\n    C조 1위 vs F조 2위\n    E조 2위 vs I조 2위\n\n    A조 1위 vs C/E/F/H/I조 3위\n    L조 1위 vs E/H/I/J/K조 3위\n\n    K조 2위 vs L조 2위\n    H조 1위 vs J조 2위\n\n    D조 1위 vs B/E/F/I/J조 3위\n    G조 1위 vs A/E/H/I/J조 3위\n\n    J조 1위 vs H조 2위\n    D조 2위 vs G조 2위\n\n    B조 1위 vs E/F/G/I/J조 3위\n    K조 1위 vs D/E/I/J/L조 3위\n]\n'

In [610]:

def do_tornament(team_df, group):
    # 1, 2, 3위 추출
    first_place = {k: v[0] for k, v in group.items()}
    second_place = {k: v[1] for k, v in group.items() if len(v) > 1}
    third_place_candidates = [(k, v[2]) for k, v in group.items() if len(v) > 2]
    
    # 3위 배정
    used_third = []
    def assign_third(allowed_groups):
        candidates = [team for team in third_place_candidates 
                      if team[0] in allowed_groups and team[1] not in used_third]
        if not candidates:
            return None
        choice = random.choice(candidates)
        used_third.append(choice[1])
        return choice[1]

    # 32강 매치업
    matches = []
    round_32 = [
        (first_place['E'], assign_third(["A","B","C","D","F","G","H","J","K","L"])),
        (first_place['I'], assign_third(["A","B","C","D","F","G","H","J","K","L"])),

        (second_place['A'], second_place['B']),
        (first_place['F'], second_place['C']),

        (first_place['C'], second_place['F']),
        (second_place['E'], second_place['I']),

        (first_place['A'], assign_third(["B","C","D","E","F","G","H","I","J","K"])),
        (first_place['L'], assign_third(["B","C","D","E","F","G","H","I","J","K"])),

        (second_place['K'], second_place['L']),
        (first_place['H'], second_place['J']),

        (first_place['D'], assign_third(["A","B","C","E","F","H","I","J","K","L"])),
        (first_place['G'], assign_third(["A","B","C","E","F","H","I","J","K","L"])),

        (first_place['J'], second_place['H']),
        (second_place['D'], second_place['G']),

        (first_place['B'], assign_third(["A","C","D","E","F","G","H","I","J","L"])),
        (first_place['K'], assign_third(["A","C","D","E","F","G","H","I","J","L"])),
    ] 


    for i, match in enumerate(round_32):
        matches.append(match)
        print(f"Match {i+1}: {match[0]} vs {match[1]}")

    print(matches)

    round_32_list = [list(t) for t in matches]
    print(round_32_list)

    flat_list = [x for t in matches for x in t]
    print(flat_list)

result_group = do_tornament(X, result_group)
result_group


Match 1: 31 vs 19
Match 2: 29 vs 17
Match 3: 3 vs 14
Match 4: 37 vs 34
Match 5: 39 vs 11
Match 6: 25 vs 16
Match 7: 10 vs 4
Match 8: 44 vs 35
Match 9: 6 vs 27
Match 10: 33 vs 24
Match 11: 42 vs 5
Match 12: 47 vs 8
Match 13: 40 vs 12
Match 14: 38 vs 20
Match 15: 30 vs 2
Match 16: 13 vs 0
[(31, 19), (29, 17), (3, 14), (37, 34), (39, 11), (25, 16), (10, 4), (44, 35), (6, 27), (33, 24), (42, 5), (47, 8), (40, 12), (38, 20), (30, 2), (13, 0)]
[[31, 19], [29, 17], [3, 14], [37, 34], [39, 11], [25, 16], [10, 4], [44, 35], [6, 27], [33, 24], [42, 5], [47, 8], [40, 12], [38, 20], [30, 2], [13, 0]]
[31, 19, 29, 17, 3, 14, 37, 34, 39, 11, 25, 16, 10, 4, 44, 35, 6, 27, 33, 24, 42, 5, 47, 8, 40, 12, 38, 20, 30, 2, 13, 0]
